# Train feature-pack v2 (Kaggle: T4 ×2 + Internet ON)

**Run All, then walk away (~1 hour).** This produces the 12 v2 walk-forward
artifacts (4 folds × 3 seeds) the promotion-gate session needs. Nothing here
touches the live site — `ARTIFACT_ROOT` stays on v1 until/unless v2 passes the
pre-registered gate.

1. Session options → Accelerator **GPU T4 ×2** (not the P100 — it's sm_60, unsupported), **Internet ON**.
2. Run All.
3. Come back to either a push confirmation or a `v2_artifacts.zip` to download
   and commit locally (Kaggle usually can't push with your git creds).

After a session cutoff, just Run All again: completed runs skip themselves and
any interrupted run auto-resumes from its checkpoint. The eval / calibration /
promotion-gate step is a **separate** session — this notebook only trains and saves.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:  # fresh Kaggle/Colab session: not inside a clone yet, so make one
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
import subprocess

# Feature-pack v2: 4 walk-forward folds x 3 seeds = 12 runs.
# seed 42 is the config default -> NO --seed flag -> lands in models/transformer/v2/.
# seeds 43/44 -> --seed suffixes run_name -> models/transformer/v2_s43, v2_s44.
# skip-if-complete makes a re-run after any session cutoff resume-only.
configs = [
    "configs/transformer_v2_through2022.yaml",
    "configs/transformer_v2_through2023.yaml",
    "configs/transformer_v2_through2024.yaml",
    "configs/transformer_v2.yaml",          # production fold, val_season 2025
]
for seed in (42, 43, 44):
    for cfg in configs:
        cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
               "--features-parquet", "data/features_2012_2025.parquet"]
        if seed != 42:                       # 42 = config default, keep run_name "v2"
            cmd += ["--seed", str(seed)]
        print()
        print(f"=== seed {seed}  {cfg} ===", flush=True)
        subprocess.run(cmd, check=True)
print()
print("All 12 v2 runs complete.")

In [ ]:
import subprocess, zipfile
from pathlib import Path

# Commit ONLY the v2 artifacts. We deliberately do NOT run the bake-off /
# promotion gate here: that is a separate, controlled evaluation session
# (pre-registered gate = PPR MAE < 4.326, p50 pinball not worse, calibration
# coverage 0.75-0.85). Committing these does not change the live v1 site --
# ARTIFACT_ROOT still points at v1 until/unless v2 passes the gate.
roots = ["models/transformer/v2",
         "models/transformer/v2_s43",
         "models/transformer/v2_s44"]

subprocess.run(["git", "add", *roots], check=False)
subprocess.run(
    ["git", "commit", "-m",
     "model: transformer v2 walk-forward artifacts (3-seed, feature-pack v2)"],
    check=False,
)
pushed = subprocess.run(["git", "push"], check=False).returncode == 0

if pushed:
    print("Pushed v2 artifacts to main. Ready for the gate session.")
else:
    # Kaggle usually can't push with your git creds -- zip for local commit.
    zip_path = Path("v2_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in roots:
            for f in Path(r).rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
    for line in [
        f"git push failed -- wrote {zip_path.resolve()} instead.",
        "Download it, then in your local clone:",
        f"  unzip {zip_path.name}",
        "  git add models/transformer/v2 models/transformer/v2_s43 models/transformer/v2_s44",
        '  git commit -m "model: transformer v2 walk-forward artifacts (3-seed, feature-pack v2)"',
        "  git push",
    ]:
        print(line)